# Inbound Quantity Forecast Training

This notebook trains a next-day inbound quantity model from WMS history.

Target: `next_day_inbound_qty` per product.

Inputs: inbound history, outbound history, current inventory, shortage records, and product metadata.

In [1]:
# If your torch_gpu environment is missing DB helper libraries, run this once:
# %pip install SQLAlchemy PyMySQL python-dotenv pandas

from pathlib import Path
from datetime import datetime
import json
import math
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

FORCE_CPU = False  # Set True when another project is already using the GPU.

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

DEVICE = torch.device('cpu' if FORCE_CPU else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print('gpu free GB:', round(free_bytes / 1024**3, 2), '/', round(total_bytes / 1024**3, 2))
DEVICE

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 3060
gpu free GB: 10.98 / 12.0


device(type='cuda')

In [2]:
PROJECT_ROOT = Path.cwd()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
MODEL_DIR = BACKEND_ROOT / 'ml' / 'inbound_forecast'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 14
BATCH_SIZE = 8
EPOCHS = 60
HIDDEN_DIM = 32
NUM_LAYERS = 1
DROPOUT = 0.0
LEARNING_RATE = 1e-3
MIN_SEQUENCES = 30

print('project root:', PROJECT_ROOT)
print('model dir:', MODEL_DIR)

project root: C:\Users\hi\Desktop\개인 프로젝트\WMS_project
model dir: C:\Users\hi\Desktop\개인 프로젝트\WMS_project\backend\ml\inbound_forecast


## Load WMS Data

The notebook reads `backend/.env` and connects to the same MySQL database used by FastAPI.

In [3]:
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv(BACKEND_ROOT / '.env')

required_env = ['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT', 'DB_NAME']
missing = [key for key in required_env if not os.getenv(key)]
if missing:
    raise RuntimeError(f'Missing DB environment variables: {missing}. Check backend/.env')

db_user = os.getenv('DB_USER')
db_password = quote_plus(os.getenv('DB_PASSWORD'))
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(
    f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}',
    pool_pre_ping=True,
)

queries = {
    'inbounds': 'SELECT inbound_id, product_id, location_id, inbound_qty, inbound_date FROM inbounds',
    'outbounds': 'SELECT outbound_id, product_id, location_id, outbound_qty, outbound_date FROM outbounds',
    'inventories': 'SELECT inventory_id, product_id, location_id, stock_qty FROM inventories',
    'shortages': 'SELECT shortage_id, product_id, location_id, requested_qty, available_qty, shortage_qty, status, created_at FROM shortages',
    'products': 'SELECT product_id, product_name, category, price FROM products',
}

tables = {name: pd.read_sql(query, engine) for name, query in queries.items()}
for name, df in tables.items():
    print(name, df.shape)

tables['inbounds'].head()

inbounds (3516, 5)
outbounds (5189, 5)
inventories (12546, 4)
shortages (1395, 8)
products (100, 4)


,inbound_id,product_id,location_id,inbound_qty,inbound_date
0,1,53,101,101,2025-09-22 19:17:00
1,2,92,182,128,2025-04-18 16:55:00
2,3,80,50,65,2025-09-03 11:21:00
3,4,71,17,178,2025-09-03 16:05:00
4,5,82,120,94,2025-01-07 14:37:00


## Build Daily Product Features

Each product gets a daily timeline. The model learns from recent inbound/outbound/shortage behavior and product metadata.

In [4]:
inbounds = tables['inbounds'].copy()
outbounds = tables['outbounds'].copy()
inventories = tables['inventories'].copy()
shortages = tables['shortages'].copy()
products = tables['products'].copy()

if inbounds.empty:
    raise ValueError('No inbound records found. Insert inbound history before training an inbound forecast model.')

inbounds['date'] = pd.to_datetime(inbounds['inbound_date']).dt.date
if not outbounds.empty:
    outbounds['date'] = pd.to_datetime(outbounds['outbound_date']).dt.date
if not shortages.empty:
    shortages['date'] = pd.to_datetime(shortages['created_at']).dt.date

date_candidates = [inbounds['date']]
if not outbounds.empty:
    date_candidates.append(outbounds['date'])
if not shortages.empty:
    date_candidates.append(shortages['date'])

min_date = min(series.min() for series in date_candidates)
max_date = max(series.max() for series in date_candidates)
dates = pd.date_range(min_date, max_date, freq='D').date

observed_products = pd.Index(inbounds['product_id'].unique())
if not outbounds.empty:
    observed_products = observed_products.union(pd.Index(outbounds['product_id'].unique()))
if not shortages.empty:
    observed_products = observed_products.union(pd.Index(shortages['product_id'].unique()))
if not products.empty:
    observed_products = observed_products.union(pd.Index(products['product_id'].unique()))

grid = pd.MultiIndex.from_product(
    [sorted(observed_products.astype(int)), dates],
    names=['product_id', 'date'],
).to_frame(index=False)

daily_inbound = (
    inbounds.groupby(['product_id', 'date'], as_index=False)
    .agg(inbound_qty=('inbound_qty', 'sum'), inbound_count=('inbound_id', 'count'))
)

if outbounds.empty:
    daily_outbound = pd.DataFrame(columns=['product_id', 'date', 'outbound_qty', 'outbound_count'])
else:
    daily_outbound = (
        outbounds.groupby(['product_id', 'date'], as_index=False)
        .agg(outbound_qty=('outbound_qty', 'sum'), outbound_count=('outbound_id', 'count'))
    )

if shortages.empty:
    daily_shortage = pd.DataFrame(columns=['product_id', 'date', 'shortage_qty', 'shortage_count'])
else:
    daily_shortage = (
        shortages.groupby(['product_id', 'date'], as_index=False)
        .agg(shortage_qty=('shortage_qty', 'sum'), shortage_count=('shortage_id', 'count'))
    )

current_stock = (
    inventories.groupby('product_id', as_index=False)
    .agg(current_stock_qty=('stock_qty', 'sum'))
    if not inventories.empty else pd.DataFrame(columns=['product_id', 'current_stock_qty'])
)

daily = grid.merge(daily_inbound, on=['product_id', 'date'], how='left')
daily = daily.merge(daily_outbound, on=['product_id', 'date'], how='left')
daily = daily.merge(daily_shortage, on=['product_id', 'date'], how='left')
daily = daily.merge(current_stock, on='product_id', how='left')
daily = daily.merge(products, on='product_id', how='left')

numeric_fill_cols = [
    'inbound_qty', 'inbound_count', 'outbound_qty', 'outbound_count',
    'shortage_qty', 'shortage_count', 'current_stock_qty', 'price',
]
for col in numeric_fill_cols:
    if col in daily.columns:
        daily[col] = daily[col].fillna(0)

daily['category'] = daily['category'].fillna('unknown').astype(str)
category_map = {value: idx for idx, value in enumerate(sorted(daily['category'].unique()))}
daily['category_code'] = daily['category'].map(category_map).astype(int)

daily['date_ts'] = pd.to_datetime(daily['date'])
daily['weekday_num'] = daily['date_ts'].dt.weekday
daily['month'] = daily['date_ts'].dt.month
daily['day'] = daily['date_ts'].dt.day
daily['is_weekend'] = (daily['weekday_num'] >= 5).astype(int)

daily = daily.sort_values(['product_id', 'date_ts']).reset_index(drop=True)

for window in [3, 7, 14]:
    daily[f'avg{window}_inbound_qty'] = (
        daily.groupby('product_id')['inbound_qty']
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
        .fillna(0)
    )
    daily[f'avg{window}_outbound_qty'] = (
        daily.groupby('product_id')['outbound_qty']
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
        .fillna(0)
    )
    daily[f'avg{window}_shortage_qty'] = (
        daily.groupby('product_id')['shortage_qty']
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
        .fillna(0)
    )

daily['prev_inbound_qty'] = daily.groupby('product_id')['inbound_qty'].shift(1).fillna(0)
daily['prev_outbound_qty'] = daily.groupby('product_id')['outbound_qty'].shift(1).fillna(0)
daily['target_next_inbound_qty'] = daily.groupby('product_id')['inbound_qty'].shift(-1)

daily = daily.dropna(subset=['target_next_inbound_qty']).reset_index(drop=True)
daily.head()

,product_id,date,inbound_qty,inbound_count,outbound_qty,outbound_count,shortage_qty,shortage_count,current_stock_qty,product_name,...,avg3_shortage_qty,avg7_inbound_qty,avg7_outbound_qty,avg7_shortage_qty,avg14_inbound_qty,avg14_outbound_qty,avg14_shortage_qty,prev_inbound_qty,prev_outbound_qty,target_next_inbound_qty
0,1,2025-01-02,0.0,0.0,0.0,0.0,0.0,0.0,87465,생수0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,52.0
1,1,2025-01-03,52.0,1.0,0.0,0.0,0.0,0.0,87465,생수0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,1,2025-01-04,0.0,0.0,0.0,0.0,0.0,0.0,87465,생수0,...,0.0,26.000000,0.0,0.0,26.000000,0.0,0.0,52.0,0.0,146.0
3,1,2025-01-05,146.0,1.0,0.0,0.0,0.0,0.0,87465,생수0,...,0.0,17.333333,0.0,0.0,17.333333,0.0,0.0,0.0,0.0,0.0
4,1,2025-01-06,0.0,0.0,23.0,1.0,0.0,0.0,87465,생수0,...,0.0,49.500000,0.0,0.0,49.500000,0.0,0.0,146.0,0.0,161.0


In [5]:
feature_cols = [
    'product_id', 'category_code', 'price', 'current_stock_qty',
    'weekday_num', 'month', 'day', 'is_weekend',
    'inbound_qty', 'inbound_count', 'outbound_qty', 'outbound_count',
    'shortage_qty', 'shortage_count',
    'prev_inbound_qty', 'prev_outbound_qty',
    'avg3_inbound_qty', 'avg7_inbound_qty', 'avg14_inbound_qty',
    'avg3_outbound_qty', 'avg7_outbound_qty', 'avg14_outbound_qty',
    'avg3_shortage_qty', 'avg7_shortage_qty', 'avg14_shortage_qty',
]

missing_features = [col for col in feature_cols if col not in daily.columns]
if missing_features:
    raise RuntimeError(f'Missing feature columns: {missing_features}')

daily[feature_cols] = daily[feature_cols].replace([np.inf, -np.inf], 0).fillna(0)
daily['target_next_inbound_qty'] = daily['target_next_inbound_qty'].clip(lower=0)

print('daily rows:', len(daily))
print('products:', daily['product_id'].nunique())
print('date range:', daily['date'].min(), 'to', daily['date'].max())
daily[['product_id', 'date', 'inbound_qty', 'outbound_qty', 'shortage_qty', 'target_next_inbound_qty']].head(10)

daily rows: 52500
products: 100
date range: 2025-01-02 to 2026-06-10


,product_id,date,inbound_qty,outbound_qty,shortage_qty,target_next_inbound_qty
0,1,2025-01-02,0.0,0.0,0.0,52.0
1,1,2025-01-03,52.0,0.0,0.0,0.0
2,1,2025-01-04,0.0,0.0,0.0,146.0
3,1,2025-01-05,146.0,0.0,0.0,0.0
4,1,2025-01-06,0.0,23.0,0.0,161.0
5,1,2025-01-07,161.0,0.0,0.0,0.0
6,1,2025-01-08,0.0,0.0,0.0,0.0
7,1,2025-01-09,0.0,0.0,0.0,140.0
8,1,2025-01-10,140.0,0.0,0.0,81.0
9,1,2025-01-11,81.0,0.0,0.0,0.0


## Create Sequence Dataset

For each product, the model sees the previous `WINDOW_SIZE` daily feature rows and predicts the next day's inbound quantity.

In [6]:
sequences = []
targets = []
sequence_meta = []

for product_id, group in daily.groupby('product_id'):
    group = group.sort_values('date_ts').reset_index(drop=True)
    x_values = group[feature_cols].to_numpy(dtype=np.float32)
    y_values = group['target_next_inbound_qty'].to_numpy(dtype=np.float32)
    for end_idx in range(WINDOW_SIZE - 1, len(group)):
        sequences.append(x_values[end_idx - WINDOW_SIZE + 1:end_idx + 1])
        targets.append(y_values[end_idx])
        sequence_meta.append({
            'product_id': int(product_id),
            'based_on_date': str(group.loc[end_idx, 'date']),
            'target_date': str(pd.to_datetime(group.loc[end_idx, 'date']) + pd.Timedelta(days=1))[:10],
        })

X = np.stack(sequences).astype(np.float32) if sequences else np.empty((0, WINDOW_SIZE, len(feature_cols)), dtype=np.float32)
y = np.array(targets, dtype=np.float32).reshape(-1, 1)

print('sequence shape:', X.shape)
print('target shape:', y.shape)

if len(X) < MIN_SEQUENCES:
    raise ValueError(
        f'Only {len(X)} training sequences were created. Need at least {MIN_SEQUENCES}. '
        'Add more inbound/outbound history or lower WINDOW_SIZE/MIN_SEQUENCES for experimentation.'
    )

sequence shape: (51200, 14, 25)
target shape: (51200, 1)


In [7]:
train_size = int(len(X) * 0.8)
X_train, X_valid = X[:train_size], X[train_size:]
y_train, y_valid = y[:train_size], y[train_size:]

class NumpyStandardScaler:
    def fit(self, values):
        values = np.asarray(values, dtype=np.float32)
        self.mean_ = values.mean(axis=0)
        self.scale_ = values.std(axis=0)
        self.scale_ = np.where(self.scale_ == 0, 1.0, self.scale_)
        return self

    def transform(self, values):
        values = np.asarray(values, dtype=np.float32)
        return (values - self.mean_) / self.scale_

    def inverse_transform(self, values):
        values = np.asarray(values, dtype=np.float32)
        return (values * self.scale_) + self.mean_

    def to_dict(self):
        return {'mean': self.mean_.tolist(), 'scale': self.scale_.tolist()}


scaler_X = NumpyStandardScaler().fit(X_train.reshape(-1, X_train.shape[-1]))
scaler_y = NumpyStandardScaler().fit(y_train)

def scale_X(values):
    original_shape = values.shape
    scaled = scaler_X.transform(values.reshape(-1, original_shape[-1]))
    return scaled.reshape(original_shape).astype(np.float32)

X_train_scaled = scale_X(X_train)
X_valid_scaled = scale_X(X_valid)
y_train_scaled = scaler_y.transform(y_train).astype(np.float32)
y_valid_scaled = scaler_y.transform(y_valid).astype(np.float32)

print('train:', X_train_scaled.shape, y_train_scaled.shape)
print('valid:', X_valid_scaled.shape, y_valid_scaled.shape)

train: (40960, 14, 25) (40960, 1)
valid: (10240, 14, 25) (10240, 1)


In [8]:
class InboundSequenceDataset(Dataset):
    def __init__(self, X_values, y_values):
        self.X_values = torch.tensor(X_values, dtype=torch.float32)
        self.y_values = torch.tensor(y_values, dtype=torch.float32)

    def __len__(self):
        return len(self.X_values)

    def __getitem__(self, idx):
        return self.X_values[idx], self.y_values[idx]


train_loader = DataLoader(
    InboundSequenceDataset(X_train_scaled, y_train_scaled),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
valid_loader = DataLoader(
    InboundSequenceDataset(X_valid_scaled, y_valid_scaled),
    batch_size=BATCH_SIZE,
    shuffle=False,
)


class InboundLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])


model = InboundLSTM(
    input_dim=len(feature_cols),
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

InboundLSTM(
  (lstm): LSTM(25, 32, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [9]:
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print('gpu free GB before training:', round(free_bytes / 1024**3, 2), '/', round(total_bytes / 1024**3, 2))

history = []
best_valid_loss = float('inf')
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    valid_losses = []
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            valid_losses.append(criterion(pred, yb).item())

    train_loss = float(np.mean(train_losses))
    valid_loss = float(np.mean(valid_losses)) if valid_losses else train_loss
    history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | train {train_loss:.5f} | valid {valid_loss:.5f}')

if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history).tail()

gpu free GB before training: 10.96 / 12.0
Epoch 001 | train 0.85700 | valid 0.43085
Epoch 010 | train 0.81506 | valid 0.43984
Epoch 020 | train 0.64984 | valid 0.47945
Epoch 030 | train 0.51968 | valid 0.46704
Epoch 040 | train 0.44963 | valid 0.50576
Epoch 050 | train 0.39936 | valid 0.50580
Epoch 060 | train 0.37199 | valid 0.53414


,epoch,train_loss,valid_loss
55,56,0.379325,0.493526
56,57,0.386036,0.504583
57,58,0.381300,0.492414
58,59,0.374595,0.524598
59,60,0.371990,0.534137


In [17]:
model.eval()
with torch.no_grad():
    valid_pred_scaled = model(torch.tensor(X_valid_scaled, dtype=torch.float32).to(DEVICE)).cpu().numpy()

valid_pred = scaler_y.inverse_transform(valid_pred_scaled).ravel()
valid_true = y_valid.ravel()
valid_pred = np.clip(np.rint(valid_pred), 0, None)

mae = float(np.mean(np.abs(valid_true - valid_pred)))
rmse = float(np.sqrt(np.mean((valid_true - valid_pred) ** 2)))

print('MAE:', round(mae, 3))
print('RMSE:', round(rmse, 3))

eval_df = pd.DataFrame(sequence_meta[train_size:]).copy()
eval_df['actual_next_inbound_qty'] = valid_true.astype(int)
eval_df['predicted_next_inbound_qty'] = valid_pred.astype(int)
eval_df.head(20)

MAE: 6.867
RMSE: 21.837


,product_id,based_on_date,target_date,actual_next_inbound_qty,predicted_next_inbound_qty
0,81,2025-01-15,2025-01-16,0,8
1,81,2025-01-16,2025-01-17,0,4
2,81,2025-01-17,2025-01-18,0,7
3,81,2025-01-18,2025-01-19,57,4
4,81,2025-01-19,2025-01-20,0,6
5,81,2025-01-20,2025-01-21,0,6
6,81,2025-01-21,2025-01-22,0,6
7,81,2025-01-22,2025-01-23,0,6
8,81,2025-01-23,2025-01-24,0,6
9,81,2025-01-24,2025-01-25,0,6


## Save Model Artifacts

Artifacts are saved to `backend/ml/inbound_forecast` so the AI server can load them later.

In [18]:
torch.save(model.state_dict(), MODEL_DIR / 'model.pt')

with (MODEL_DIR / 'scaler_X.json').open('w', encoding='utf-8') as file:
    json.dump(scaler_X.to_dict(), file, ensure_ascii=False, indent=2)
with (MODEL_DIR / 'scaler_y.json').open('w', encoding='utf-8') as file:
    json.dump(scaler_y.to_dict(), file, ensure_ascii=False, indent=2)

metadata = {
    'model_type': 'InboundLSTM',
    'target': 'next_day_inbound_qty',
    'features': feature_cols,
    'window_size': WINDOW_SIZE,
    'input_dim': len(feature_cols),
    'category_map': category_map,
    'metrics': {'mae': float(mae), 'rmse': float(rmse)},
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'device': str(DEVICE),
}

with (MODEL_DIR / 'metadata.json').open('w', encoding='utf-8') as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

daily.to_csv(MODEL_DIR / 'training_frame.csv', index=False, encoding='utf-8-sig')
eval_df.to_csv(MODEL_DIR / 'validation_predictions.csv', index=False, encoding='utf-8-sig')

print('saved:', MODEL_DIR)
print(sorted(path.name for path in MODEL_DIR.iterdir()))

saved: C:\Users\hi\Desktop\개인 프로젝트\WMS_project\backend\ml\inbound_forecast
['metadata.json', 'model.pt', 'scaler_X.json', 'scaler_y.json', 'training_frame.csv', 'validation_predictions.csv']


## Quick Inference Check

Use this helper to preview the next inbound quantity for one product after training.

In [16]:
def predict_next_inbound(product_id: int):
    product_frame = daily[daily['product_id'] == product_id].sort_values('date_ts').copy()
    if len(product_frame) < WINDOW_SIZE:
        raise ValueError(f'Product {product_id} has only {len(product_frame)} rows. Need {WINDOW_SIZE}.')

    seq = product_frame[feature_cols].tail(WINDOW_SIZE).to_numpy(dtype=np.float32)[None, :, :]
    seq_scaled = scale_X(seq)

    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.tensor(seq_scaled, dtype=torch.float32).to(DEVICE)).cpu().numpy()

    pred = scaler_y.inverse_transform(pred_scaled)[0, 0]
    based_on_date = pd.to_datetime(product_frame['date'].iloc[-1]).date()
    target_date = based_on_date + pd.Timedelta(days=1)

    return {
        'product_id': int(product_id),
        'predicted_inbound_qty': max(0, int(np.rint(pred))),
        'predicted_inbound_qty_raw': float(pred),
        'based_on_date': based_on_date.isoformat(),
        'target_date': target_date.isoformat(),
    }

sample_product_id = int(daily['product_id'].iloc[0])
predict_next_inbound(sample_product_id)

{'product_id': 1,
 'predicted_inbound_qty': 0,
 'predicted_inbound_qty_raw': -0.3839435577392578,
 'based_on_date': '2026-06-10',
 'target_date': '2026-06-11'}